# 11 — Hybrid Aggregation Predictor

Combines ESM-2 embeddings (PCA-reduced) + flow model structural features + ESM-2 LLR
into a single GradientBoosting model for aggregation rate prediction.

**Goal:** Bridge the performance gap between:
- Flow ΔRg only: ρ ≈ 0.23
- ESM-2 embedding probe: ρ ≈ 0.66
- **Hybrid target: ρ ≥ 0.45**

**Runtime:** ~15 min on Colab T4 (mostly flow ensemble generation)

In [ ]:
# CELL 0: Setup
import os, sys
os.chdir('/content')
if not os.path.exists('brain_idp_flow'):
    !git clone https://github.com/layeredlabs06/brain-idp-flow.git brain_idp_flow
else:
    !cd brain_idp_flow && git pull origin master
os.chdir('brain_idp_flow')
!pip install -e . -q
!pip install fair-esm scipy scikit-learn openpyxl -q
sys.path.insert(0, '/content/brain_idp_flow/src')

import torch, numpy as np, yaml, pickle, os
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

In [ ]:
# CELL 1: Drive 마운트 + 모델/캐시 로드
from google.colab import drive
drive.mount('/content/drive')

from brain_idp_flow.sample import load_model, sample_ensemble
from brain_idp_flow.features.seq_embed import ESM2Embedder
from brain_idp_flow.geometry.metrics import radius_of_gyration
from brain_idp_flow.data.dataset import ProteinEnsembleDataset
from brain_idp_flow.data.dms_loader import load_seuma_dms, ABETA42_WT
from brain_idp_flow.analysis.esm2_llr import ESM2MutationScorer
from brain_idp_flow.model.hybrid_predictor import EnsembleFeatureExtractor, HybridAggregationPredictor

CKPT_35M = '/content/drive/MyDrive/brain_idp_flow_ckpts/ckpt_best.pt'
SCORE_CACHE = '/content/drive/MyDrive/brain_idp_flow_scored/all_scores.pkl'

with open('configs/flow.yaml') as f:
    config = yaml.safe_load(f)

model = load_model(config, CKPT_35M, device)
embedder = ESM2Embedder(device=device)
scorer = ESM2MutationScorer(device=device)
print('Flow model + ESM-2 loaded')

AA_TO_IDX = {aa: i + 1 for i, aa in enumerate('ACDEFGHIKLMNPQRSTVWY')}

In [ ]:
# CELL 2: DMS 로드 + ESM-2 임베딩 + LLR (Drive 캐시에서 LLR 가져오기)
os.makedirs('data/dms', exist_ok=True)
dms_data = load_seuma_dms(cache_dir='data/dms')
print(f'Loaded {len(dms_data)} Aβ42 mutations')

# Drive 캐시에서 기존 scored_dms 로드 (LLR 값 포함)
scored_dms = None
if os.path.exists(SCORE_CACHE):
    with open(SCORE_CACHE, 'rb') as f:
        cached = pickle.load(f)
    if 'scored_dms' in cached:
        scored_dms = cached['scored_dms']
        print(f'Loaded scored_dms from Drive cache ({len(scored_dms)} items)')

# ESM-2 임베딩 (mean-pooled)
print('Computing ESM-2 embeddings...')
mut_embeddings = []
for d in dms_data:
    seq = list(ABETA42_WT)
    seq[d['pos'] - 1] = d['mt']
    mut_seq = ''.join(seq)
    emb = embedder.embed_single(mut_seq)
    mut_embeddings.append(emb.mean(dim=0).cpu().numpy())
mut_embeddings = np.array(mut_embeddings)
print(f'Embeddings: {mut_embeddings.shape}')

# LLR — 캐시가 있으면 재사용, 없으면 새로 계산
if scored_dms and all('llr_site' in d for d in scored_dms):
    llr_values = np.array([d['llr_site'] for d in scored_dms])
    print(f'LLR loaded from cache')
else:
    print('Computing ESM-2 LLR...')
    llr_values = []
    for d in dms_data:
        result = scorer.score_mutation(ABETA42_WT, d['pos'], d['wt'], d['mt'], fast=True)
        llr_values.append(result['llr_site'])
    llr_values = np.array(llr_values)

rho_llr, p_llr = spearmanr(llr_values, [d['nucleation_score'] for d in dms_data])
print(f'ESM-2 LLR vs nucleation: ρ={rho_llr:.3f}, p={p_llr:.4f}')

## 2. Compute ESM-2 Embeddings

In [ ]:
# (merged into Cell 2 — ESM-2 embeddings computed there)

## 3. Compute ESM-2 LLR

In [ ]:
# (merged into Cell 2 — LLR computed there)

## 4. Generate Flow Ensembles & Extract Structural Features

This is the slow step (~10 min). We generate WT + each mutant ensemble,
then extract geometry features.

In [ ]:
# CELL 3: Flow 앙상블 생성 + 구조 피처 추출
from tqdm.auto import tqdm

extractor = EnsembleFeatureExtractor()

# WT 앙상블
wt_emb = embedder.embed_single(ABETA42_WT)
wt_ensemble = sample_ensemble(
    model, wt_emb, mut_pos=0, mut_aa=0,
    n_samples=64, n_steps=50, device=device, batch_size=32,
)
wt_rg = radius_of_gyration(torch.from_numpy(wt_ensemble)).mean().item()
print(f'WT ensemble: {wt_ensemble.shape}, Rg={wt_rg:.2f}')

# 돌연변이별 앙상블 + 구조 피처
structural_features = []
for i, d in enumerate(tqdm(dms_data, desc='Generating mutant ensembles')):
    mut_seq = list(ABETA42_WT)
    mut_seq[d['pos'] - 1] = d['mt']
    mut_seq = ''.join(mut_seq)
    mut_emb = embedder.embed_single(mut_seq)
    mut_aa_idx = AA_TO_IDX.get(d['mt'], 0)

    mut_ensemble = sample_ensemble(
        model, mut_emb, mut_pos=d['pos'], mut_aa=mut_aa_idx,
        n_samples=32, n_steps=50, device=device, batch_size=32,
    )

    sf = extractor.extract(wt_ensemble, mut_ensemble, d['pos'])
    structural_features.append(sf)

print(f'Extracted features for {len(structural_features)} mutations')
print(f'Feature keys: {list(structural_features[0].keys())}')

In [ ]:
# (merged into Cell 3)

## 5. Train Hybrid Model (5-fold CV)

In [ ]:
# CELL 4: 하이브리드 모델 학습 (5-fold CV)
targets = np.array([d['nucleation_score'] for d in dms_data])

predictor = HybridAggregationPredictor(n_pca=50, n_folds=5)
results = predictor.fit(
    embeddings=mut_embeddings,
    structural_features=structural_features,
    llr_values=llr_values,
    targets=targets,
)

## 6. Ablation: Compare Model Variants

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

def cv_rho(X, y, n_folds=5):
    """Quick 5-fold CV Spearman rho for GBRegressor."""
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    rhos = []
    for tr, te in kf.split(X):
        sc = StandardScaler()
        gb = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.1,
                                       min_samples_leaf=5, random_state=42)
        gb.fit(sc.fit_transform(X[tr]), y[tr])
        pred = gb.predict(sc.transform(X[te]))
        r, _ = spearmanr(pred, y[te])
        if not np.isnan(r):
            rhos.append(r)
    return np.mean(rhos) if rhos else 0.0

# Prepare feature variants
pca = PCA(n_components=min(50, len(dms_data)-1), random_state=42)
emb_pca = pca.fit_transform(mut_embeddings)

struct_keys = list(structural_features[0].keys())
struct_matrix = np.array([[sf[k] for k in struct_keys] for sf in structural_features])

y = targets

# Ablation table
variants = {
    'ΔRg only': struct_matrix[:, struct_keys.index('delta_rg')].reshape(-1, 1),
    'LLR only': llr_values.reshape(-1, 1),
    'Structural only': struct_matrix,
    'Embedding PCA only': emb_pca,
    'Embedding + LLR': np.hstack([emb_pca, llr_values.reshape(-1, 1)]),
    'Structural + LLR': np.hstack([struct_matrix, llr_values.reshape(-1, 1)]),
    'Hybrid (all)': np.hstack([emb_pca, struct_matrix, llr_values.reshape(-1, 1)]),
}

print(f"{'Variant':<25} {'CV ρ':>8} {'Features':>10}")
print('-' * 45)
for name, X_var in variants.items():
    rho = cv_rho(X_var, y)
    print(f'{name:<25} {rho:>8.3f} {X_var.shape[1]:>10}')

In [ ]:
# CELL 5b: Bootstrap 95% CI for ablation results (fast version)
# 각 variant를 한번만 fit하고, bootstrap은 prediction vs target의 Spearman에만 적용

def fast_bootstrap_ci(X, y, n_boot=10000, n_folds=5, seed=42):
    """Fit model once with CV, then bootstrap the correlation."""
    from sklearn.model_selection import KFold
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    y_pred = np.zeros_like(y)
    for tr, te in kf.split(X):
        sc = StandardScaler()
        gb = GradientBoostingRegressor(n_estimators=100, max_depth=3,
                                       learning_rate=0.1, min_samples_leaf=5, random_state=seed)
        gb.fit(sc.fit_transform(X[tr]), y[tr])
        y_pred[te] = gb.predict(sc.transform(X[te]))

    rho_obs, _ = spearmanr(y_pred, y)

    # Bootstrap the correlation
    rng = np.random.RandomState(seed)
    n = len(y)
    boot_rhos = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        r, _ = spearmanr(y_pred[idx], y[idx])
        if not np.isnan(r):
            boot_rhos.append(r)
    boot_rhos = np.sort(boot_rhos)
    lo = boot_rhos[int(0.025 * len(boot_rhos))]
    hi = boot_rhos[int(0.975 * len(boot_rhos))]
    return rho_obs, lo, hi

print(f"{'Variant':<25} {'CV ρ':>8} {'95% CI':>20} {'Features':>10}")
print('-' * 65)
ablation_results = {}
for name, X_var in variants.items():
    rho, lo, hi = fast_bootstrap_ci(X_var, y)
    ablation_results[name] = {'rho': rho, 'ci_lo': lo, 'ci_hi': hi, 'p': X_var.shape[1]}
    print(f'{name:<25} {rho:>8.3f} [{lo:>.3f}, {hi:>.3f}]{X_var.shape[1]:>10}')

In [ ]:
# CELL 5c: Ablation figure (for paper)
os.makedirs('results/hybrid', exist_ok=True)

names = list(ablation_results.keys())
rhos = [ablation_results[n]['rho'] for n in names]
ci_los = [ablation_results[n]['ci_lo'] for n in names]
ci_his = [ablation_results[n]['ci_hi'] for n in names]
errors_lo = [r - lo for r, lo in zip(rhos, ci_los)]
errors_hi = [hi - r for r, hi in zip(rhos, ci_his)]

# Sort by rho
order = np.argsort(rhos)
names = [names[i] for i in order]
rhos = [rhos[i] for i in order]
errors_lo = [errors_lo[i] for i in order]
errors_hi = [errors_hi[i] for i in order]

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d73027' if 'Embedding' in n or 'Hybrid' in n else '#4575b4' for n in names]
bars = ax.barh(range(len(names)), rhos, xerr=[errors_lo, errors_hi],
               color=colors, alpha=0.8, capsize=4, edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=10)
ax.set_xlabel('5-fold CV Spearman ρ', fontsize=12)
ax.set_title('Hybrid Model Ablation: Feature Set Comparison\n(A β42 DMS, n=751)', fontsize=13)
ax.axvline(x=0, color='gray', linestyle='--', linewidth=0.5)

# Annotate rho values
for i, (r, lo, hi) in enumerate(zip(rhos, errors_lo, errors_hi)):
    ax.text(r + hi + 0.01, i, f'{r:.3f}', va='center', fontsize=9)

fig.tight_layout()
fig.savefig('results/hybrid/ablation_barplot.png', dpi=300, bbox_inches='tight')
fig.savefig('results/hybrid/ablation_barplot.pdf', bbox_inches='tight')
plt.show()
print('Saved: results/hybrid/ablation_barplot.png + .pdf')

# Copy to Drive
try:
    import shutil
    shutil.copy('results/hybrid/ablation_barplot.png',
                '/content/drive/MyDrive/brain_idp_flow_results/hybrid_ablation.png')
    print('Copied to Drive')
except:
    pass

## 7. IAPP Out-of-Distribution Validation

Train on Aβ42 DMS, evaluate on IAPP mutations from targets.yaml.

In [ ]:
# CELL 6: OOD 검증 (α-synuclein or tau)
from brain_idp_flow.targets import load_targets

targets_cfg = load_targets('configs/targets.yaml')
iapp_target = None
for tid, t in targets_cfg.items():
    if tid in ('asyn', 'tau_k18'):
        iapp_target = t
        break

if iapp_target is not None:
    print(f'OOD target: {iapp_target.name} ({len(iapp_target.mutations)} mutations)')

    ood_embeddings = []
    ood_llr = []
    ood_struct = []
    ood_targets = []

    wt_seq = iapp_target.sequence
    wt_emb_ood = embedder.embed_single(wt_seq)
    wt_ens_ood = sample_ensemble(
        model, wt_emb_ood, mut_pos=0, mut_aa=0,
        n_samples=64, n_steps=50, device=device
    )

    for mut in tqdm(iapp_target.mutations, desc=f'OOD: {iapp_target.name}'):
        if mut.agg_rate_relative is None:
            continue

        mut_seq = iapp_target.mutant_sequence(mut)
        mut_emb_ood = embedder.embed_single(mut_seq)
        ood_embeddings.append(mut_emb_ood.mean(dim=0).cpu().numpy())

        result = scorer.score_mutation(wt_seq, mut.pos, mut.wt, mut.mt, fast=True)
        ood_llr.append(result['llr_site'])

        mut_aa_idx = AA_TO_IDX.get(mut.mt, 0)
        mut_ens = sample_ensemble(
            model, mut_emb_ood, mut_pos=mut.pos, mut_aa=mut_aa_idx,
            n_samples=32, n_steps=50, device=device, batch_size=32
        )
        sf = extractor.extract(wt_ens_ood, mut_ens, mut.pos)
        ood_struct.append(sf)
        ood_targets.append(np.log(mut.agg_rate_relative + 1e-8))

    if ood_targets:
        ood_result = predictor.evaluate_ood(
            embeddings=np.array(ood_embeddings),
            structural_features=ood_struct,
            llr_values=np.array(ood_llr),
            targets=np.array(ood_targets),
            dataset_name=iapp_target.name,
        )
else:
    print('No suitable OOD target found in targets.yaml')

## 8. Feature Contribution Visualization

In [ ]:
# CELL 9: Summary + Export embedding-only predictor
from brain_idp_flow.model.embedding_predictor import EmbeddingAggregationPredictor

# Train embedding-only predictor
emb_predictor = EmbeddingAggregationPredictor(n_pca=50)
emb_results = emb_predictor.fit(mut_embeddings, targets)

# Save to Drive
SAVE_PATH = '/content/drive/MyDrive/brain_idp_flow_scored/embedding_predictor.pkl'
emb_predictor.save(SAVE_PATH)

print('\n' + '='*60)
print('RESULTS SUMMARY')
print('='*60)
print(f"{'Method':<30} {'Aβ42 CV ρ':>12}")
print('-'*45)
print(f"{'Flow ΔRg only':<30} {'~0.23':>12}")
print(f"{'Cross-scale (GB)':<30} {'~0.35':>12}")
print(f"{'Embedding PCA-50 (GB)':<30} {emb_results['cv_mean_rho']:>12.3f}")
print(f"\nModel saved to: {SAVE_PATH}")
print(f"PCA variance explained: {emb_results['pca_variance_explained']:.1%}")
print('\nTo use locally:')
print('  1. Download embedding_predictor.pkl from Drive')
print('  2. MODEL_PATH=embedding_predictor.pkl python -m brain_idp_flow.app')
print('\nDone!')

## 9. Summary Table

In [ ]:
print('\n' + '='*60)
print('RESULTS SUMMARY')
print('='*60)
print(f"{'Method':<30} {'Aβ42 CV ρ':>12} {'OOD ρ':>10}")
print('-'*55)
print(f"{'Flow ΔRg only':<30} {'~0.23':>12} {'~0.11':>10}")
print(f"{'ESM-2 embedding probe':<30} {'~0.66':>12} {'~0.53':>10}")
print(f"{'Cross-scale (GB)':<30} {'~0.35':>12} {'~0.12':>10}")
if results:
    ood_rho = f"{ood_result['rho']:.3f}" if 'ood_result' in dir() and ood_result else 'N/A'
    print(f"{'Hybrid (this notebook)':<30} {results['cv_mean_rho']:>12.3f} {ood_rho:>10}")
print('\nDone!')